# Agile Deliberation Demo

Licensed under the Apache License, Version 2.0.

In [ ]:
# @title Dependencies

%load_ext autoreload
%autoreload 2

import logging
import os
import pickle
import slugify
import warnings

import numpy as np

from IPython.display import HTML, display

from agile_deliberation_lib import utils as utils
from agile_deliberation_lib import image as image_py
from agile_deliberation_lib import retrieval as retrieval_py
from agile_deliberation_lib import definitions as definitions_py
from agile_deliberation_lib import annotator as annotator_py
from agile_deliberation_lib import llm as llm_py
from agile_deliberation_lib import components as components_py
from agile_deliberation_lib import classifier as classifier_py
from agile_deliberation_lib import search_images as search_images_py
from agile_deliberation_lib import refine_definition as refine_definition_py
from agile_deliberation_lib import reflection as reflection_py
from agile_deliberation_lib import interaction as interaction_py
from agile_deliberation_lib import diverse_sample as diverse_sample_py
from agile_deliberation_lib import deliberators as deliberators_py

from agile_deliberation_lib.retrieval import RetrievalClient

warnings.filterwarnings("ignore", category=RuntimeWarning)


# automatically wrap long lines of text in the output
def set_css(info=None):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)


Definition = definitions_py.Definition
MyImage = image_py.MyImage


logging.basicConfig(
    encoding='utf-8',
    level=logging.WARNING,
    force=True,
    format='%(levelname)s - %(funcName)s - %(lineno)d: %(message)s'
)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('PIL').setLevel(logging.WARNING)
logging.getLogger('google.ai.generativelanguage').setLevel(logging.WARNING)

concept_deliberator = None

In [ ]:
#@title Setup the retrieval client using nearest neighbor.

# Point to the indices path specified in the indices_train.json file,
# which can be found in the code repository.
retrieval_client = retrieval_py.RetrievalClient(indices_path="indices_train.json")

In [ ]:
# @title Setup the LLM and test if it works

# @markdown Provide an API key here:
api_key = ''  # @param {type:"string"}

# @markdown Customize the model names if needed:
expensive_model_name = 'gemini-2.5-pro'  # @param {type:"string"}
cheap_model_name = 'gemini-2.5-flash'  # @param {type:"string"}

model_client = llm_py.ModelClient(
    cheap_model_names=[cheap_model_name] if cheap_model_name else None,
    api_key=api_key if api_key else None,
    expensive_model_name=expensive_model_name if expensive_model_name else llm_py.DEFAULT_MODEL_NAME,
)

# Run this to test whether these basic functionalities work;
NUM_IMAGES = 3

images = retrieval_client.text_image_search('pandas', NUM_IMAGES)
print(f'There are in total {len(images)} images.')
for i in range(min(NUM_IMAGES, len(images))):
  images[i].show()
  print(model_client.gemini_image_prompt('What is in this image, answer in one sentence?', [images[i]]))

# ***Experiment Tasks***

# Instructions

Imagine you are a moderator for a Reddit community about healthy food. Your goal is to approve **all images that show healthy and nutritious food**. We encourage you to have a slightly higher standard as there is limited space to showcase these images in your community.

Here are a few in-scope examples,

<div style="display:flex; gap:8px; align-items:flex-start">
<img src="https://cdn.prod.website-files.com/63bf3f41c3bc69578d823f01/6696f09f8741e2a1e4afeb05_plate%20of%20food.jpg" alt="Plate of food" style="height:200px"/>
<img src="https://healthylittlevittles.com/wp-content/uploads/2022/02/Anti-Inflammatory-Vegan-Green-Smoothie-4.jpg" alt="Green smoothie" style="height:200px"/>
</div>

and a few out-of-scope examples.

The first image is out of scope because it contains fries, while the second image depicts smoothies with heavy cream.

<div style="display:flex; gap:8px; align-items:flex-start">
<img src="https://www.mccain.co.uk/media/y4aitvlg/crispy-salmon-zesty-home-chips.jpg" alt="Fries" style="height:150px"/>
<img src="https://www.centercutcook.com/wp-content/uploads/2024/05/cookies-and-cream-smoothie.jpg" alt="Cream smoothie" style="height:150px"/>
<img src="https://natashaskitchen.com/wp-content/uploads/2024/03/Caesar-Salad-Dressing-copy.jpg" alt="Caesar dressing" style="height:150px"/>
<img src="https://www.flyinggoosebrand.com/wp-content/uploads/2022/09/Sriracha-mayo-sauce-can-be-enjoyed-on-nearly-any-food-including-sushi.-1.jpg" alt="Sriracha mayo" style="height:150px"/>
</div>

In this notebook, you will help refine the definition of this concept, with the flexibility to address ambiguous scenarios that the initial description might not fully capture. Your definition will then be used as prompts for a language model to determine whether an image is in-scope or out-of-scope for this concept.

In [ ]:
#@markdown ## **Start with a simple description**
#@markdown We will start with a simple description of this concept.

#@markdown **Do not run this cell twice**
experiment_folder = "./outputs" # @param {type: "string"}
concept = 'Healthy Food' #@param {type:"string"}

slug_concept = slugify.slugify(concept)
experiment_name = 'concept_deliberation'
output_dir = f"{experiment_folder}/{slug_concept}/{experiment_name}"
os.makedirs(output_dir, exist_ok=True)

# Initialize ConceptDeliberator
concept_deliberator = deliberators_py.ConceptDeliberator(
    retrieval_client, model_client, definition_folder=output_dir, keep_output=False)
display_with_styles = components_py.display_with_styles

# @markdown Initial concept definition:
concept_description = "Images that show healthy food."  # @param {type:"string"}
definition = Definition(concept, concept_description)

# ---------------------------------------------------------------------------
# Pre-filled signal categories so Step 1 runs without LLM calls.
# Each round shows 2 golden + 1 borderline category (3 total per round).
# 4 golden + 2 borderline covers two full rounds.
# Remove this block to let the LLM brainstorm its own categories.
# This is useful if you want to test the iteration stage, and do not repeatedly run the scoping stage, or if you are low on llm credits.
# ---------------------------------------------------------------------------
concept_deliberator.interaction.reflection.cached_concepts['Healthy Food'] = [
    {
        'concept': 'Healthy Food',
        'description': 'Images that show healthy food.',
        'golden_categories': [
            {
                'concept': 'Healthy Dish',
                'description': (
                    'Images show a prepared meal or dish that is prominently '
                    'composed of healthy ingredients, such as salads, grain bowls, '
                    'grilled or steamed lean proteins (e.g., chicken breast, fish, '
                    'or tofu) with vegetables, or oatmeal with fruit.'
                ),
            },
            {
                'concept': 'Healthy Beverages',
                'description': (
                    'Images show healthy beverages, such as smoothies made from '
                    'fruits and vegetables, freshly squeezed juices, or '
                    'fruit-infused water.'
                ),
            },
            {
                'concept': 'Fresh Produce',
                'description': (
                    'Images show fresh fruits or vegetables presented as food, '
                    'such as a fruit bowl, a vegetable platter, or a colourful '
                    'arrangement of seasonal produce ready to eat.'
                ),
            },
            {
                'concept': 'Healthy Snacks',
                'description': (
                    'Images show healthy snack options such as mixed nuts, seeds, '
                    'hummus with vegetable sticks, or homemade energy bars made '
                    'from whole ingredients.'
                ),
            },
        ],
        'borderline_categories': [
            {
                'concept': 'Processed Food',
                'description': (
                    'Images show processed foods typically considered unhealthy, '
                    'such as pizza, deep-fried items, mayonnaise, or whipped cream.'
                ),
            },
            {
                'concept': 'Raw Ingredients or Non-Food Focus',
                'description': (
                    'Images show raw uncooked meat or seafood, farming or '
                    'harvesting scenes, grocery aisles, or activities such as '
                    'cooking or dining out where the food itself is not the '
                    'main subject.'
                ),
            },
        ],
    }
]

In [ ]:
#@markdown ## **Step 1: Enrich Your Concept Definitions**
#@markdown Our system will begin by enriching the concept definition you've provided.

#@markdown You are asked to determine whether each signal is in-scope or out-of-scope.
#@markdown Reviewing relevant images can help you get a better idea of what this signal refers to.

concept_deliberator.enrich_definitions(definition)

Output()

In [ ]:
#@markdown ## **Step 2: Run this cell to review your resultant definition**

#@markdown As you create a sketch of your concept definition at the first stage,
#@markdown you are invited to spend two minutes reviewing your current definition.

#@markdown In particular, some signals might be repetitive, while others might no longer make sense to you.
#@markdown You could also adjust the language of each signal to make it more accurate here.

#@markdown If you want to add a new signal, also feel free to add it by clicking the button "Add New Signal".
display_with_styles(components_py.interactive_definition(definition))

In [ ]:
#@markdown ## **[Optional] Pre-collect Images for Step 3**
#@markdown Run this cell **once** to collect ~500 candidate images from the index and
#@markdown cache them to disk.  Step 3 will then load the cache automatically,
#@markdown skipping the slower on-the-fly collection.
#@markdown
#@markdown Skip this cell if you are happy to let Step 3 collect images on demand.

datasets_dir = f'{experiment_folder}/{slug_concept}/datasets'
pre_collected_path = f'{datasets_dir}/cached_images_for_iterations.pkl'

if os.path.exists(pre_collected_path):
    print(f'Cache already exists at {pre_collected_path} — skipping collection.')
    with open(pre_collected_path, 'rb') as f:
        _cached = pickle.load(f)
    print(f'{len(_cached)} images in cache.')
else:
    os.makedirs(datasets_dir, exist_ok=True)
    print('Collecting images — this may take a few minutes …')
    cached_images = concept_deliberator.prepare_images_for_reflection(
        definition, total_images_num=500
    )
    print(f'Collected {len(cached_images)} images.')
    with open(pre_collected_path, 'wb') as f:
        pickle.dump(cached_images, f)
    print(f'Saved to {pre_collected_path}')


In [ ]:
#@markdown ## **Step 3: Reflect on Borderline Images**
#@markdown Now that you have a clear outline of your concept definition, it's important to ensure that the language is free of ambiguities.
#@markdown We will present five images in each round for you to review and label.

#@markdown **If the classifier makes a different decision from you, you are encouraged to communicate your thoughts in more detail.**

#@markdown **Otherwise, you could also write your thoughts for images that you find reflecting important ambiguities.**

#@markdown Our system will then try its best to edit the definition according to your feedback.

#@markdown Finally, please bear with us as the classifier is still warming up its telepathy to predict your choices.

# ---------------------------------------------------------------------------
# Option A: use the pre-collected dataset of ~500 images.
# This skips the FAISS search and image download step, saving several minutes.
# Set reflect_images_path to the path of the pre-collected pickle file.
# Recommended if you are testing the later part of the stage.
# ---------------------------------------------------------------------------
datasets_dir = f'{experiment_folder}/{slug_concept}/datasets'
pre_collected_path = f'{datasets_dir}/cached_images_for_iterations.pkl'

if os.path.exists(pre_collected_path):
    reflect_images_path = pre_collected_path
    print(f'Using pre-collected dataset: {reflect_images_path}')
else:
    # Option B: collect images on the fly from the FAISS index.
    # This takes a few minutes depending on the index size.
    reflect_images_path = None
    print('Pre-collected dataset not found — images will be collected on the fly.')

log_path = None

concept_deliberator.image_reflection(
    definition,
    reflect_images_path=reflect_images_path,
    log_path=log_path,
)